# Populate product_v with product embeddings

Run the setup, population, and verification cells in order. Set `database_name` and `source_container_name` to the existing Cosmos DB database/container containing your products. The original lab uses `test_db_5/product`; if yours is `products`, change the source name. An empty source is not automatically seeded from `products.json`.

Creating the target container only configures vector storage and indexing. The population cell calls Azure OpenAI for each product and writes the returned vector into `contentVector`, preserving the product fields, ID, and categoryId. It leaves the source products unchanged. Rerunning regenerates embeddings and upserts the target documents, which consumes embedding tokens and Cosmos DB request units.

The deployed embedding model must return 1536 dimensions to match this policy. If you change dimensions, use a matching deployment and a new target container with the matching vector policy.


In [3]:
import json
import os
import time
from pathlib import Path
from typing import Any

from dotenv import load_dotenv
from azure.cosmos import CosmosClient, PartitionKey
from azure.identity import DefaultAzureCredential
from azure.keyvault.secrets import SecretClient
from openai import AzureOpenAI

loaded = load_dotenv()
if not loaded:
    raise RuntimeError(".env file could not be loaded.")

key_vault_url = os.environ["KEY_VAULT_URL"]
secret_name = os.environ["COSMOS_DB_CONNECTION_STRING_SECRET_NAME"]

credential = DefaultAzureCredential()
secret_client = SecretClient(vault_url=key_vault_url, credential=credential)
connection_string = secret_client.get_secret(secret_name).value

if connection_string is None:
    raise RuntimeError(f"Key Vault secret '{secret_name}' has no value.")

client = CosmosClient.from_connection_string(connection_string)

# Point these names at the EXISTING database and container holding your products.
database_name = "test_db_5"
source_container_name = "product"  # Use "products" if that is your container name.
target_container_name = "product_v"
if source_container_name == target_container_name:
    raise ValueError("Source and target containers must be different.")
database = client.get_database_client(database_name)
source_products_container = database.get_container_client(source_container_name)
source_products_container.read()  # Fail clearly if the source container does not exist.

# Create a document
# source_products_container.upsert_item({"id": "1", "categoryId": "electronics"})

# items = ContainerProxy.query_items(query="SELECT * FROM product", enable_cross_partition_query=True)

# Add product data to database using batch operations
# Get cosmic works product data from github
# product_raw_data = "https://cosmosdbcosmicworks.blob.core.windows.net/cosmic-works-small/product.json"
# product_data = ProductList(items=[Product(**data) for data in requests.get(product_raw_data).json()])
# for prod in product_data.items:
#     print(prod)
#     source_products_container.upsert_item(prod.model_dump(by_alias=True))

# product_raw_data = "https://cosmosdbcosmicworks.blob.core.windows.net/cosmic-works-small/product.json"
# product_data = ProductList(items=[Product(**data) for data in requests.get(product_raw_data).json()])
# for prod in product_data.items:
#     source_products_container.upsert_item(prod.model_dump(by_alias=True))

# product_file = Path("products.json")
# with product_file.open(encoding="utf-8") as file:
#     product_data = ProductList(
#         items=[Product(**data) for data in json.load(file)]
#     )

# for prod in product_data.items:
#     source_products_container.upsert_item(prod.model_dump(by_alias=True))

aoai_endpoint = os.environ["AOAI_ENDPOINT"]
api_version = os.environ["AOAI_API_VERSION"]
aoai_key = os.environ["AOAI_KEY"]

# Instantiate an AzureOpenAI client
ai_client = AzureOpenAI(
    azure_endpoint=aoai_endpoint,
    api_version=api_version,
    api_key=aoai_key,
)

# Generate embeddings from a text string
embeddings_deployment_name = os.environ["EMBEDDINGS_DEPLOYMENT_NAME"]


def generate_embeddings(text: str):
    '''
    Generate embeddings from string of text using the deployed Azure OpenAI API embeddings model.
    This will be used to vectorize document data and incoming user messages for a similarity search with
    the vector index.
    '''
    if not text.strip():
        raise ValueError("Embedding input must contain text.")
    response = ai_client.embeddings.create(
        input=text, model=embeddings_deployment_name, encoding_format="float"
    )
    embeddings = response.data[0].embedding
    expected_dimensions = vector_embedding_policy["vectorEmbeddings"][0]["dimensions"]
    if len(embeddings) != expected_dimensions:
        raise ValueError(
            f"Embedding has {len(embeddings)} dimensions; the container policy expects "
            f"{expected_dimensions}. Use a matching deployment and vector policy."
        )
    time.sleep(0.5)  # rest period to avoid rate limiting on AOAI
    return embeddings


# Create the vector embedding policy
vector_embedding_policy = {
    "vectorEmbeddings": [
        {
            "path": "/contentVector",
            "dataType": "float32",
            "distanceFunction": "cosine",
            "dimensions": 1536,
        }
    ]
}

# Create the indexing policy
indexing_policy = {
    "indexingMode": "consistent",
    "automatic": True,
    "includedPaths": [{"path": "/*"}],
    "excludedPaths": [
        {"path": "/\"_etag\"/?"},
        {"path": "/contentVector/*"},
    ],
    "vectorIndexes": [{"path": "/contentVector", "type": "diskANN"}],
}

product_v_container = database.create_container_if_not_exists(
    id=target_container_name,
    partition_key=PartitionKey(path="/categoryId"),
    indexing_policy=indexing_policy,
    vector_embedding_policy=vector_embedding_policy,
)

# create_container_if_not_exists does not update an existing container's policies.
target_properties = product_v_container.read()
if target_properties.get("partitionKey", {}).get("paths") != ["/categoryId"]:
    raise ValueError("The target container must use /categoryId as its partition key.")
if target_properties.get("vectorEmbeddingPolicy") != vector_embedding_policy:
    raise ValueError(
        "The target has a different vector policy. Choose a new target_container_name "
        "with the desired policy before generating embeddings."
    )


ModuleNotFoundError: No module named 'dotenv'

## Generate and store embeddings


In [ ]:
# Read existing Cosmos DB products, generate embeddings, and persist enriched copies.
# Reruns upsert the same id/categoryId pairs rather than creating duplicates.
print("API version:", api_version)
print("Deployment:", embeddings_deployment_name)
print("Dimensions:", len(generate_embeddings("test product")))

written = 0
skipped = 0
products = source_products_container.query_items(
    query="SELECT * FROM c", enable_cross_partition_query=True
)
for product in products:
    if not product.get("id") or not product.get("categoryId"):
        raise ValueError("Every source product needs an id and categoryId.")
    text = "\n".join(
        f"{field}: {product[field]}"
        for field in ("name", "description", "categoryName")
        if product.get(field) and str(product[field]).strip()
    )
    if not text:
        skipped += 1
        print(f"Skipping {product['id']}: no name, description, or categoryName.")
        continue
    # Cosmos DB supplies these system properties; don't copy them to the target.
    document = {
        key: value for key, value in product.items()
        if key not in {"_rid", "_self", "_etag", "_attachments", "_ts", "contentVector"}
    }
    document["contentVector"] = generate_embeddings(text)
    product_v_container.upsert_item(body=document)
    written += 1
    print(f"Stored {document['id']} ({len(document['contentVector'])} dimensions)")

if written == 0:
    raise RuntimeError(
        f"No products were written. Check {database_name}/{source_container_name} "
        f"contains products with text. Skipped {skipped} products."
    )
print(f"Finished: wrote {written} products to {target_container_name}; skipped {skipped}.")


## Verify persisted embeddings


In [ ]:
# Verify a persisted vector without printing the whole array.
samples = list(product_v_container.query_items(
    query="SELECT TOP 1 c.id, c.name, c.contentVector FROM c "
          "WHERE IS_ARRAY(c.contentVector)",
    enable_cross_partition_query=True,
))
if not samples or not samples[0]["contentVector"]:
    raise RuntimeError("No stored embeddings found in the target container.")
sample = samples[0]
print({
    "id": sample["id"],
    "name": sample.get("name"),
    "dimensions": len(sample["contentVector"]),
    "first_five_values": sample["contentVector"][:5],
})


## Vector search and RAG helpers

Use `list(vector_search(product_v_container, "electronics"))` after running this cell. The RAG helper additionally requires `query_item_by_id`, `system_prompt`, and `deployment_name` from the rest of the guide.


In [ ]:
### Performing a vector search query
def query_item_by_id(container, item_id: str, model_cls: type[Product] | None = None):
    """Read a full document from Cosmos DB and convert it to a product model if needed."""
    doc = container.read_item(item=item_id, partition_key=item_id)
    doc.pop("contentVector", None)
    if model_cls is None:
        return doc
    if hasattr(model_cls, "model_validate"):
        return model_cls.model_validate(doc)
    return model_cls(**doc)


system_prompt = """You are a helpful assistant for a product catalog. Use only the product information in the retrieved context to answer user questions. If the answer is not present, say you do not know. Keep responses concise and factual."""

deployment_name = os.environ.get("CHAT_DEPLOYMENT_NAME") or os.environ.get("AOAI_CHAT_DEPLOYMENT_NAME") or os.environ.get("EMBEDDINGS_DEPLOYMENT_NAME")


def vector_search(
        container,
        prompt: str,
        vector_field_name: str = "contentVector",
        num_results: int = 5):
    query_embedding = generate_embeddings(prompt)
    items = container.query_items(
        query=f"""SELECT TOP @num_results itm.id, VectorDistance(itm.{vector_field_name}, @embedding) AS SimilarityScore 
                FROM itm
                ORDER BY VectorDistance(itm.{vector_field_name}, @embedding)
                """,
        parameters=[
            {"name": "@num_results", "value": num_results},
            {"name": "@embedding", "value": query_embedding},
        ],
        enable_cross_partition_query=True,
    )
    return items


### Using vector search results with an LLM in a RAG scenario
def rag_with_vector_search(
        container,
        prompt: str,
        vector_field_name: str = "contentVector",
        num_results: int = 5):
    """
    Use the RAG model to generate a prompt using vector search results based on the
    incoming question.
    """
    results = vector_search(container, prompt, vector_field_name, num_results)
    product_list = ""
    for result in results:
        product = query_item_by_id(container, result["id"], Product)
        product_doc = product.model_dump(exclude_none=True, by_alias=True) if hasattr(product, "model_dump") else dict(product)
        product_doc.pop("contentVector", None)
        product_list += json.dumps(product_doc, default=str, indent=4) + "\n\n"

    formatted_prompt = system_prompt + "\n\n" + product_list
    messages = [
        {"role": "system", "content": formatted_prompt},
        {"role": "user", "content": prompt},
    ]
    completion = ai_client.chat.completions.create(messages=messages, model=deployment_name)
    return completion.choices[0].message.content


In [ ]:
# Run this cell after the vector_search helper above. Edit the query to suit your catalog.
search_text = "lightweight product for hiking"
results = list(vector_search(product_v_container, search_text, num_results=5))

print(f"Top {len(results)} matches for: {search_text!r}")
for rank, item in enumerate(results, start=1):
    print(f"{rank}. id={item['id']} | distance={item['SimilarityScore']:.4f}")
if not results:
    print("No matches found. Check that product_v contains embedded products.")


## Reference: LangChain and agent examples

These original examples are kept as reference text because they depend on additional setup from the guide (`CONNECTION_STRING`, `embedding_model`, `llm`, and other helpers). They are not needed to populate `product_v`; configure their database/container names to use your target before adapting them.

```python
### Instantiating a vector store reference

from azure.cosmos import CosmosClient
from langchain_azure_ai.vectorstores.azure_cosmos_db_no_sql import (
    AzureCosmosDBNoSqlVectorSearch,
)

cosmos_client = CosmosClient.from_connection_string(CONNECTION_STRING)

vector_store = AzureCosmosDBNoSqlVectorSearch(
    cosmos_client = cosmos_client,
    database_name = "cosmic_works",
    container_name = "products",
    embedding = embedding_model,
    vector_embedding_policy = vector_embedding_policy,
    indexing_policy = indexing_policy
)

# Create a retriever from the vector store
retriever = vector_store.as_retriever()

# Create the prompt template from the system_prompt text
llm_prompt = PromptTemplate.from_template(system_prompt)

rag_chain = (
    # populate the tokens/placeholders in the llm_prompt
    # products takes the results of the vector store and formats the documents
    # question is a passthrough that takes the incoming question
    { "products": retriever | format_docs, "question": RunnablePassthrough()}
    | llm_prompt
    # pass the populated prompt to the language model
    | llm
    # return the string ouptut from the language model
    | StrOutputParser()
)

# create a chain on the retriever to format the documents as JSON
products_retriever_chain = products_retriever | format_docs

tools = [
    Tool(
        name = "vector_search_products",
        func = products_retriever_chain.invoke,
        description = "Searches Cosmic Works product information for similar products based on the question. Returns the product information in JSON format."
    )
]

container = CosmosClient(CONNECTION_STRING).get_database_client("cosmic_works").get_container_client("products")

def get_product_by_id(product_id: str) -> str:
    """
    Retrieves a product by its ID.
    """
    doc = container.read_item(item=product_id, partition_key=product_id)
    if "contentVector" in doc:
        del doc["contentVector"]
    return json.dumps(doc)

def get_product_by_sku(sku: str) -> str:
    """
    Retrieves a product by its sku.
    """
    query = f"SELECT * FROM c WHERE c.sku = '{sku}'"
    results = list(container.query_items(query=query, enable_cross_partition_query=True))
    if results:
        doc = results[0]
        if "contentVector" in doc:
            del doc["contentVector"]
        return json.dumps(doc, default=str)
    return "{}"

from langchain.tools import StructuredTool

tools.extend([
    StructuredTool.from_function(get_product_by_id),
    StructuredTool.from_function(get_product_by_sku)
    #StructuredTool.from_function(get_sales_by_id)
])

### Creating an agent armed with tools for vector search and Python functions calling

system_message = SystemMessage(
    content = """
        You are a helpful, fun and friendly sales assistant for Cosmic Works, a bicycle and bicycle accessories store.

        Your name is Cosmo.

        You are designed to answer questions about the products that Cosmic Works sells, the customers that buy them, and the sales orders that are placed by customers.

        If you don't know the answer to a question, respond with "I don't know."

        Only answer questions related to Cosmic Works products, customers, and sales orders.

        If a question is not related to Cosmic Works products, customers, or sales orders,
        respond with "I only answer questions about Cosmic Works"
    """
)
agent_executor = create_conversational_retrieval_agent(llm, tools, system_message = system_message, verbose=True)
```
